# Legal AI Platform Analytics

Interactive dashboard notebook — fetches live data from the SvelteKit dev server APIs and visualizes platform metrics.

**Prerequisites**: Dev server running at `http://localhost:5173` with `DEV_BYPASS_AUTH=true`

In [ ]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from datetime import datetime

BASE_URL = 'http://localhost:5173'
PALETTE = {
    'sand': '#d4c7a3', 'blue': '#60a5fa', 'green': '#4ade80',
    'red': '#f87171', 'amber': '#fbbf24', 'purple': '#a78bfa',
    'teal': '#34d399', 'bg': '#0e0d0b', 'panel': '#1e1d19',
}
DARK_TEMPLATE = 'plotly_dark'

session = requests.Session()
session.cookies.set('dev_bypass', 'true')

def api_get(path: str, params: dict = None) -> dict | None:
    """Fetch JSON from a local API endpoint."""
    try:
        r = session.get(f'{BASE_URL}{path}', params=params, timeout=10)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'[WARN] {path}: {e}')
        return None

print(f'Notebook ready — targeting {BASE_URL}')
print(f'Timestamp: {datetime.now().isoformat()}')

## 1. Dashboard Stats Overview

Core platform counts: cases, evidence, POIs, citations, and knowledge base entries.

In [ ]:
dash = api_get('/api/dashboard/stats')

if dash:
    summary = {
        'Active Cases': dash.get('activeCases', 0),
        'Total Evidence': dash.get('totalEvidence', 0),
        'Persons of Interest': dash.get('personsOfInterest', 0),
        'Citations': dash.get('totalCitations', 0),
        'Saved Citations': dash.get('savedCitations', 0),
    }
    kb = dash.get('knowledgeBase', {})
    summary.update({
        'KB: Glossary': kb.get('glossary', 0),
        'KB: Statutes': kb.get('statutes', 0),
        'KB: Precedents': kb.get('precedents', 0),
    })

    df_dash = pd.DataFrame(list(summary.items()), columns=['Metric', 'Count'])
    display(df_dash)

    fig = px.bar(
        df_dash, x='Metric', y='Count',
        color='Count', color_continuous_scale=[[0, PALETTE['panel']], [1, PALETTE['blue']]],
        template=DARK_TEMPLATE,
        title='Platform Overview — Key Metrics',
    )
    fig.update_layout(
        paper_bgcolor=PALETTE['bg'], plot_bgcolor=PALETTE['panel'],
        font_color=PALETTE['sand'], xaxis_tickangle=-30, height=400,
        coloraxis_showscale=False,
    )
    fig.show()
else:
    print('Dashboard stats unavailable — is the dev server running?')

## 2. Analytics Deep Dive (30-day window)

Case status breakdown, error brain health, and fix rate gauge.

In [ ]:
analytics = api_get('/api/yorha/analytics', {'period': '30d'})

if analytics:
    cases = analytics.get('cases', {})
    errors = analytics.get('errorBrain', {})

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{'type': 'pie'}, {'type': 'bar'}, {'type': 'indicator'}]],
        subplot_titles=['Case Status', 'Error Brain', 'Fix Rate'],
    )

    # Case status donut
    other = max(0, cases.get('total', 0) - cases.get('active', 0) - cases.get('closed', 0))
    fig.add_trace(go.Pie(
        labels=['Active', 'Closed', 'Other'],
        values=[cases.get('active', 0), cases.get('closed', 0), other],
        marker_colors=[PALETTE['blue'], PALETTE['green'], '#525045'],
        hole=0.5, textinfo='label+value',
    ), row=1, col=1)

    # Error brain horizontal bar
    unfixed = max(0, errors.get('totalErrors', 0) - errors.get('fixed', 0))
    fig.add_trace(go.Bar(
        x=[errors.get('fixed', 0), unfixed],
        y=['Fixed', 'Open'],
        orientation='h',
        marker_color=[PALETTE['green'], PALETTE['red']],
        text=[errors.get('fixed', 0), unfixed],
        textposition='auto',
    ), row=1, col=2)

    # Fix rate gauge
    fig.add_trace(go.Indicator(
        mode='gauge+number+delta',
        value=errors.get('fixRate', 100),
        number={'suffix': '%'},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': PALETTE['green']},
            'steps': [
                {'range': [0, 50], 'color': 'rgba(248,113,113,0.2)'},
                {'range': [50, 80], 'color': 'rgba(251,191,36,0.2)'},
                {'range': [80, 100], 'color': 'rgba(74,222,128,0.2)'},
            ],
        },
    ), row=1, col=3)

    fig.update_layout(
        template=DARK_TEMPLATE,
        paper_bgcolor=PALETTE['bg'], plot_bgcolor=PALETTE['panel'],
        font_color=PALETTE['sand'], height=400, showlegend=False,
        title_text=f'Analytics — {analytics.get("period", "30d")} window',
    )
    fig.show()
else:
    print('Analytics API unavailable')

## 3. Knowledge Base Composition

Treemap showing the relative size of glossary terms, statutes, precedents, and citations.

In [ ]:
if dash:
    kb = dash.get('knowledgeBase', {})
    kb_data = [
        {'Category': 'Glossary', 'Count': kb.get('glossary', 0), 'Parent': 'Knowledge Base'},
        {'Category': 'Statutes', 'Count': kb.get('statutes', 0), 'Parent': 'Knowledge Base'},
        {'Category': 'Precedents', 'Count': kb.get('precedents', 0), 'Parent': 'Knowledge Base'},
        {'Category': 'Citations', 'Count': dash.get('totalCitations', 0), 'Parent': 'Knowledge Base'},
        {'Category': 'Saved Citations', 'Count': dash.get('savedCitations', 0), 'Parent': 'Knowledge Base'},
    ]
    df_kb = pd.DataFrame(kb_data)
    df_kb = df_kb[df_kb['Count'] > 0]  # filter zeros for cleaner treemap

    if not df_kb.empty:
        fig = px.treemap(
            df_kb, path=['Parent', 'Category'], values='Count',
            color='Count',
            color_continuous_scale=[
                [0, PALETTE['panel']], [0.5, PALETTE['purple']], [1, PALETTE['blue']]
            ],
            template=DARK_TEMPLATE,
            title='Knowledge Base Composition',
        )
        fig.update_layout(
            paper_bgcolor=PALETTE['bg'], font_color=PALETTE['sand'], height=400,
            coloraxis_showscale=False,
        )
        fig.show()
    else:
        print('No knowledge base entries yet')
else:
    print('Dashboard stats unavailable')

## 4. Infrastructure Health Matrix

Service availability heatmap with latency overlay.

In [ ]:
infra = api_get('/api/infrastructure/status')

if infra:
    # Extract service statuses from nested response
    svc_list = []
    status_map = {'up': 1, 'healthy': 1, 'degraded': 0.5, 'down': 0, 'unavailable': 0}

    def extract_services(obj, prefix=''):
        if isinstance(obj, dict):
            for k, v in obj.items():
                key = f'{prefix}.{k}' if prefix else k
                if isinstance(v, dict) and 'status' in v:
                    svc_list.append({
                        'Service': key,
                        'Status': v['status'],
                        'Score': status_map.get(str(v['status']).lower(), 0),
                        'Latency': v.get('latencyMs', v.get('latency_ms', None)),
                    })
                elif isinstance(v, str) and v.lower() in status_map:
                    svc_list.append({'Service': key, 'Status': v, 'Score': status_map.get(v.lower(), 0), 'Latency': None})
                elif isinstance(v, dict):
                    extract_services(v, key)

    extract_services(infra)

    if svc_list:
        df_svc = pd.DataFrame(svc_list)
        display(df_svc[['Service', 'Status', 'Latency']].head(20))

        fig = px.bar(
            df_svc.head(15), x='Score', y='Service', orientation='h',
            color='Score',
            color_continuous_scale=[
                [0, PALETTE['red']], [0.5, PALETTE['amber']], [1, PALETTE['green']]
            ],
            template=DARK_TEMPLATE,
            title='Infrastructure Health — Service Status',
            labels={'Score': 'Health (0=down, 1=up)'},
        )
        fig.update_layout(
            paper_bgcolor=PALETTE['bg'], plot_bgcolor=PALETTE['panel'],
            font_color=PALETTE['sand'], height=max(300, len(df_svc) * 28),
            xaxis_range=[0, 1.1], coloraxis_showscale=False,
            yaxis={'categoryorder': 'total ascending'},
        )
        fig.show()
    else:
        print('Could not parse infrastructure services')
else:
    print('Infrastructure API unavailable')

## 5. Cache Performance

Redis cache hit rates, key counts, and L1 exact-match stats.

In [ ]:
cache = api_get('/api/cache/stats')
l1_cache = api_get('/api/cache/exact-match/stats')

cache_metrics = []
if cache:
    cache_metrics.append({'Layer': 'Redis (General)', **{k: v for k, v in cache.items() if isinstance(v, (int, float, str))}})
if l1_cache:
    cache_metrics.append({'Layer': 'L1 Exact-Match', **{k: v for k, v in l1_cache.items() if isinstance(v, (int, float, str))}})

if cache_metrics:
    df_cache = pd.DataFrame(cache_metrics)
    display(df_cache)

    # Hit rate gauges
    fig = make_subplots(
        rows=1, cols=len(cache_metrics),
        specs=[[{'type': 'indicator'}] * len(cache_metrics)],
        subplot_titles=[m['Layer'] for m in cache_metrics],
    )

    for i, m in enumerate(cache_metrics, 1):
        hit_rate = m.get('hitRate', m.get('hit_rate', m.get('hitRatePercent', 0)))
        if isinstance(hit_rate, str):
            try: hit_rate = float(hit_rate.replace('%', ''))
            except: hit_rate = 0

        fig.add_trace(go.Indicator(
            mode='gauge+number',
            value=float(hit_rate),
            number={'suffix': '%'},
            gauge={
                'axis': {'range': [0, 100]},
                'bar': {'color': PALETTE['blue']},
                'steps': [
                    {'range': [0, 50], 'color': 'rgba(248,113,113,0.15)'},
                    {'range': [50, 80], 'color': 'rgba(251,191,36,0.15)'},
                    {'range': [80, 100], 'color': 'rgba(74,222,128,0.15)'},
                ],
            },
        ), row=1, col=i)

    fig.update_layout(
        template=DARK_TEMPLATE,
        paper_bgcolor=PALETTE['bg'], font_color=PALETTE['sand'],
        height=300, title_text='Cache Hit Rates',
    )
    fig.show()
else:
    print('Cache stats unavailable')

## 6. Evidence Pipeline

Evidence composition and recent upload activity.

In [ ]:
if analytics:
    ev = analytics.get('evidence', {})
    total = ev.get('total', 0)
    with_files = ev.get('withFiles', 0)
    without_files = max(0, total - with_files)

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'pie'}, {'type': 'bar'}]],
        subplot_titles=['Evidence: Files vs Metadata-Only', 'Evidence Activity'],
    )

    fig.add_trace(go.Pie(
        labels=['With Files', 'Metadata Only'],
        values=[with_files, without_files],
        marker_colors=[PALETTE['teal'], '#525045'],
        hole=0.45, textinfo='label+percent',
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=['Total', 'Recent Uploads', 'Cases w/ Evidence'],
        y=[total, ev.get('recentUploaded', 0), ev.get('casesWithEvidence', 0)],
        marker_color=[PALETTE['blue'], PALETTE['green'], PALETTE['purple']],
        text=[total, ev.get('recentUploaded', 0), ev.get('casesWithEvidence', 0)],
        textposition='auto',
    ), row=1, col=2)

    fig.update_layout(
        template=DARK_TEMPLATE,
        paper_bgcolor=PALETTE['bg'], plot_bgcolor=PALETTE['panel'],
        font_color=PALETTE['sand'], height=400, showlegend=False,
        title_text='Evidence Pipeline Overview',
    )
    fig.show()
else:
    print('Analytics API unavailable')

## 7. Persons of Interest — Risk Pyramid

Breakdown by threat level: critical, high, and other.

In [ ]:
if analytics:
    poi = analytics.get('personsOfInterest', {})
    critical = poi.get('critical', 0)
    high = poi.get('high', 0)
    other = max(0, poi.get('total', 0) - critical - high)

    fig = go.Figure(go.Funnel(
        y=['Other / Low', 'High', 'Critical'],
        x=[other, high, critical],
        marker_color=[PALETTE['sand'], PALETTE['amber'], PALETTE['red']],
        textinfo='value+percent total',
    ))

    fig.update_layout(
        template=DARK_TEMPLATE,
        paper_bgcolor=PALETTE['bg'], plot_bgcolor=PALETTE['panel'],
        font_color=PALETTE['sand'], height=350,
        title_text=f'Persons of Interest — Risk Pyramid ({poi.get("total", 0)} total)',
    )
    fig.show()
else:
    print('Analytics API unavailable')

## 8. System Summary

Consolidated table of all fetched data for quick reference.

In [ ]:
summary_rows = []

if analytics:
    c = analytics.get('cases', {})
    e = analytics.get('evidence', {})
    eb = analytics.get('errorBrain', {})
    p = analytics.get('personsOfInterest', {})
    sys = analytics.get('system', {})

    summary_rows.extend([
        {'Section': 'Cases', 'Metric': 'Total', 'Value': c.get('total', 0)},
        {'Section': 'Cases', 'Metric': 'Active', 'Value': c.get('active', 0)},
        {'Section': 'Cases', 'Metric': 'Closed', 'Value': c.get('closed', 0)},
        {'Section': 'Cases', 'Metric': 'Recent (30d)', 'Value': c.get('recentCreated', 0)},
        {'Section': 'Evidence', 'Metric': 'Total', 'Value': e.get('total', 0)},
        {'Section': 'Evidence', 'Metric': 'Recent Uploads', 'Value': e.get('recentUploaded', 0)},
        {'Section': 'Evidence', 'Metric': 'With Files', 'Value': e.get('withFiles', 0)},
        {'Section': 'Errors', 'Metric': 'Total', 'Value': eb.get('totalErrors', 0)},
        {'Section': 'Errors', 'Metric': 'Fixed', 'Value': eb.get('fixed', 0)},
        {'Section': 'Errors', 'Metric': 'Fix Rate', 'Value': f"{eb.get('fixRate', 100)}%"},
        {'Section': 'POI', 'Metric': 'Total', 'Value': p.get('total', 0)},
        {'Section': 'POI', 'Metric': 'Critical', 'Value': p.get('critical', 0)},
        {'Section': 'POI', 'Metric': 'High', 'Value': p.get('high', 0)},
    ])

    for svc, status in sys.items():
        summary_rows.append({'Section': 'System', 'Metric': svc.title(), 'Value': status})

if dash:
    kb = dash.get('knowledgeBase', {})
    summary_rows.extend([
        {'Section': 'Knowledge Base', 'Metric': 'Glossary', 'Value': kb.get('glossary', 0)},
        {'Section': 'Knowledge Base', 'Metric': 'Statutes', 'Value': kb.get('statutes', 0)},
        {'Section': 'Knowledge Base', 'Metric': 'Precedents', 'Value': kb.get('precedents', 0)},
        {'Section': 'Knowledge Base', 'Metric': 'Citations', 'Value': dash.get('totalCitations', 0)},
    ])

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))
    print(f'\nReport generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
else:
    print('No data available — start the dev server first')